# DA6401 — W&B Report: All Sections (2.1–2.8)

Generates all plots, tables, and visualizations for the W&B report.

**BEFORE RUNNING:** GPU T4 x2, Internet ON, Save & Run All

In [ ]:
import os
os.environ["WANDB_API_KEY"] = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT = "/kaggle/working/checkpoints"
os.makedirs(CKPT, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git /kaggle/working/repo 2>&1 | tail -3
%cd /kaggle/working/repo
!pip install -q wandb albumentations gdown scikit-learn

import torch, numpy as np, re, gdown
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from io import BytesIO
from PIL import Image as PILImage
import wandb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Download checkpoints
for did, name in [
    ('1x_qgIKPFwu_2inuIjyms7IhQY2Qr_ZKP', 'classifier.pth'),
    ('1jRM83Xr6HETheUhv1MQqRMEl2eAwbkSK', 'localizer.pth'),
    ('1ATvVvuMPagLhP3wLjI-sWe3D1Z4wHV0Z', 'unet.pth'),
]:
    out = f'{CKPT}/{name}'
    gdown.download(id=did, output=out, quiet=True, fuzzy=True)
    print(f'  {name}: {os.path.getsize(out)/1e6:.0f} MB')

# Download dataset
from data.pets_dataset import download_oxford_pet, create_dataloaders, IMAGENET_MEAN, IMAGENET_STD, OxfordIIITPetDataset
download_oxford_pet('./data/oxford_pet')
train_loader, val_loader, _ = create_dataloaders(root='./data/oxford_pet', batch_size=16, num_workers=2)
print('Dataset ready.')

In [ ]:
def _canonicalize_checkpoint(sd):
    if not any(re.match(r'encoder\.block\d+\.\d+\.\d+\.', k) for k in sd):
        return sd
    new_sd = {}
    for k, v in sd.items():
        m = re.match(r'encoder\.block(\d+)\.(\d+)\.(\d+)\.(.*)', k)
        if m:
            N, M, L, suf = int(m.group(1)), int(m.group(2)), int(m.group(3)), m.group(4)
            X = N - 1
            if X in (0, 1): Y = 0 if L == 0 else 1
            else: Y = (0 if L == 0 else 1) if M == 0 else (3 if L == 0 else 4)
            new_sd[f'encoder.block{X}.{Y}.{suf}'] = v
        elif k.startswith('classifier.'): new_sd['head.' + k[len('classifier.'):]] = v
        elif k.startswith('regressor.'): new_sd['head.' + k[len('regressor.'):]] = v
    return new_sd

from models.classification import VGG11Classifier
from models.localization import VGG11Localizer
from models.segmentation import VGG11UNet
from train import denormalize_image, compute_iou, compute_dice_score, compute_pixel_accuracy
import torch.nn as nn

cls_model = VGG11Classifier(num_classes=37, dropout_p=0.5, use_bn=True).to(device)
loc_model = VGG11Localizer(image_size=224).to(device)
seg_model = VGG11UNet(num_classes=3).to(device)

cls_model.load_state_dict(_canonicalize_checkpoint(
    torch.load(f'{CKPT}/classifier.pth', map_location=device, weights_only=False)), strict=False)
loc_model.load_state_dict(_canonicalize_checkpoint(
    torch.load(f'{CKPT}/localizer.pth', map_location=device, weights_only=False)), strict=False)
seg_model.load_state_dict(
    torch.load(f'{CKPT}/unet.pth', map_location=device, weights_only=False), strict=False)

cls_model.eval(); loc_model.eval(); seg_model.eval()
class_names = OxfordIIITPetDataset.CLASS_NAMES
print('Models loaded.')

In [ ]:
# Pull all run histories from W&B
api = wandb.Api()
PROJECT = 'da25m020-indian-institute-of-technology-madras/da6401-assignment2'
all_runs = api.runs(PROJECT)
print(f'Found {len(all_runs)} runs.')

def find_runs(tag=None, name_contains=None):
    results = []
    for r in all_runs:
        if tag and tag not in (r.tags or []): continue
        if name_contains and name_contains not in r.name: continue
        results.append(r)
    return results

# Helper to save figure to wandb image
def fig_to_wandb(fig, caption=''):
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    buf.seek(0); plt.close(fig)
    return wandb.Image(PILImage.open(buf), caption=caption)

# Start report run
run = wandb.init(project='da6401-assignment2',
                 name='full_report_panels',
                 tags=['report'])

---
## Section 2.1 — BatchNorm: Activation Distributions & Convergence

In [ ]:
# 2.1: BatchNorm effect — activation distributions at 3rd conv layer
# Run a forward pass to get activations from block1 (3rd conv = block2[0])
activations = {}
def make_hook(name):
    def fn(m, i, o): activations[name] = o.detach().cpu()
    return fn

# Hook on 3rd conv layer output (block2, first conv = encoder.block2[0])
cls_model.encoder.block2[0].register_forward_hook(make_hook('conv3_bn'))

# Get a sample image
sample_batch = next(iter(val_loader))
sample_img = sample_batch['image'][:1].to(device)

with torch.no_grad():
    _ = cls_model(sample_img)
act_bn = activations['conv3_bn'].flatten().numpy()

# Also create a no-BN model with random weights for comparison
cls_no_bn = VGG11Classifier(num_classes=37, dropout_p=0.5, use_bn=False).to(device)
cls_no_bn.eval()
cls_no_bn.encoder.block2[0].register_forward_hook(make_hook('conv3_nobn'))
with torch.no_grad():
    _ = cls_no_bn(sample_img)
act_nobn = activations['conv3_nobn'].flatten().numpy()

# Plot activation distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(act_bn, bins=100, alpha=0.7, color='#2196F3', density=True)
axes[0].set_title(f'With BatchNorm\nmean={act_bn.mean():.3f}, std={act_bn.std():.3f}', fontsize=12)
axes[0].set_xlabel('Activation Value'); axes[0].set_ylabel('Density')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)

axes[1].hist(act_nobn, bins=100, alpha=0.7, color='#FF5722', density=True)
axes[1].set_title(f'Without BatchNorm\nmean={act_nobn.mean():.3f}, std={act_nobn.std():.3f}', fontsize=12)
axes[1].set_xlabel('Activation Value'); axes[1].set_ylabel('Density')
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5)

plt.suptitle('Section 2.1: Activation Distribution at 3rd Conv Layer', fontsize=14, fontweight='bold')
plt.tight_layout()
wandb.log({'sec2.1/activation_distributions': fig_to_wandb(fig)})

# Pull BN vs no-BN training curves
bn_runs = find_runs(tag='exp-batchnorm')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for r in bn_runs:
    h = r.history(pandas=True)
    if h.empty: continue
    label = 'BN=True' if 'bn=True' in r.name else 'BN=False'
    color = '#2196F3' if 'bn=True' in r.name else '#FF5722'
    if 'train/loss' in h.columns:
        axes[0].plot(h['train/loss'].dropna().values, label=f'{label} (train)', linestyle='--', color=color, alpha=0.6)
    if 'val/loss' in h.columns:
        axes[0].plot(h['val/loss'].dropna().values, label=f'{label} (val)', color=color)
    for col in ['val/f1_macro', 'val/accuracy']:
        if col in h.columns:
            axes[1].plot(h[col].dropna().values, label=label, color=color)
            break

axes[0].set_title('Train vs Val Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Validation Metric'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
plt.suptitle('Section 2.1: BatchNorm vs No BatchNorm', fontsize=14, fontweight='bold')
plt.tight_layout()
wandb.log({'sec2.1/bn_comparison_curves': fig_to_wandb(fig)})
print('Section 2.1 done.')

---
## Section 2.2 — Dropout: Generalization Gap

In [ ]:
# 2.2: Overlay train vs val loss for drop=0.0, 0.2, 0.5
dropout_runs = find_runs(tag='exp-dropout')
colors = {'drop=0.0': '#4CAF50', 'drop=0.2': '#FF9800', 'drop=0.5': '#2196F3'}

# Pick one run per dropout value (prefer 120 epochs, 2 GPUs)
best_runs = {}
for r in dropout_runs:
    if 'bn=True' not in r.name: continue
    for dp in ['drop=0.0', 'drop=0.2', 'drop=0.5']:
        if dp in r.name:
            cfg_epochs = r.config.get('epochs', 0)
            if dp not in best_runs or cfg_epochs > best_runs[dp].config.get('epochs', 0):
                best_runs[dp] = r

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for dp, r in sorted(best_runs.items()):
    h = r.history(pandas=True)
    if h.empty: continue
    c = colors.get(dp, 'gray')
    p_val = dp.split('=')[1]
    if 'train/loss' in h.columns:
        axes[0].plot(h['train/loss'].dropna().values, label=f'p={p_val} (train)', linestyle='--', color=c, alpha=0.6)
    if 'val/loss' in h.columns:
        axes[0].plot(h['val/loss'].dropna().values, label=f'p={p_val} (val)', color=c)
    for col in ['val/f1_macro', 'val/accuracy']:
        if col in h.columns:
            axes[1].plot(h[col].dropna().values, label=f'p={p_val}', color=c)
            break

axes[0].set_title('Train vs Val Loss (Generalization Gap)'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Validation F1 / Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
plt.suptitle('Section 2.2: Custom Dropout Regularization (p=0.0, 0.2, 0.5)', fontsize=14, fontweight='bold')
plt.tight_layout()
wandb.log({'sec2.2/dropout_comparison': fig_to_wandb(fig)})
print('Section 2.2 done.')

---
## Section 2.3 — Transfer Learning: Frozen vs Partial vs Full

In [ ]:
# 2.3: Transfer learning comparison
freeze_colors = {'frozen': '#F44336', 'partial': '#FF9800', 'full': '#4CAF50'}
freeze_runs = {}
for mode in ['frozen', 'partial', 'full']:
    candidates = find_runs(tag=f'freeze-{mode}')
    if candidates:
        freeze_runs[mode] = candidates[0]  # take first match

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for mode, r in freeze_runs.items():
    h = r.history(pandas=True)
    if h.empty: continue
    c = freeze_colors[mode]
    # Loss
    if 'train/loss' in h.columns:
        axes[0].plot(h['train/loss'].dropna().values, label=f'{mode} (train)', linestyle='--', color=c, alpha=0.6)
    if 'val/loss' in h.columns:
        axes[0].plot(h['val/loss'].dropna().values, label=f'{mode} (val)', color=c)
    # Dice
    for col in ['val/dice', 'val/mean_dice']:
        if col in h.columns:
            axes[1].plot(h[col].dropna().values, label=mode, color=c)
            break
    # Pixel Accuracy
    for col in ['val/pixel_accuracy']:
        if col in h.columns:
            axes[2].plot(h[col].dropna().values, label=mode, color=c)
            break

axes[0].set_title('Train vs Val Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Validation Dice Score'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Dice')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
axes[2].set_title('Validation Pixel Accuracy'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Accuracy')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)
plt.suptitle('Section 2.3: Transfer Learning — Frozen vs Partial vs Full', fontsize=14, fontweight='bold')
plt.tight_layout()
wandb.log({'sec2.3/transfer_learning': fig_to_wandb(fig)})

# Summary table
print(f'{"Mode":10s} | {"Final Dice":>10s} | {"Final PixAcc":>12s} | {"Trainable":>10s}')
for mode, r in freeze_runs.items():
    s = r.summary
    dice = s.get('val/dice', s.get('val/mean_dice', '-'))
    pa = s.get('val/pixel_accuracy', '-')
    tp = r.config.get('trainable_params', '-')
    print(f'{mode:10s} | {dice if isinstance(dice,str) else f"{dice:.4f}":>10s} | {pa if isinstance(pa,str) else f"{pa:.4f}":>12s} | {tp}')
print('Section 2.3 done.')

---
## Section 2.4 — Feature Map Visualization

In [ ]:
# 2.4: Feature maps from first and last conv layers
activations.clear()
cls_model.encoder.block0[0].register_forward_hook(make_hook('first_conv'))
last_conv_idx = max(i for i, m in enumerate(cls_model.encoder.block4) if isinstance(m, nn.Conv2d))
cls_model.encoder.block4[last_conv_idx].register_forward_hook(make_hook('last_conv'))

with torch.no_grad():
    _ = cls_model(sample_img)

# Plot input image + feature maps
for layer_name, n_show in [('first_conv', 16), ('last_conv', 32)]:
    feat = activations[layer_name][0]
    n_ch = min(n_show, feat.shape[0])
    n_cols = 8; n_rows = (n_ch + n_cols - 1) // n_cols + 1  # +1 for input image
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*2, n_rows*2))
    axes_flat = axes.flatten()

    # Show input image in first cell
    axes_flat[0].imshow(denormalize_image(sample_img[0]))
    axes_flat[0].set_title('Input', fontsize=8, fontweight='bold')
    axes_flat[0].axis('off')
    for j in range(1, n_cols): axes_flat[j].axis('off')

    for c in range(n_ch):
        fm = feat[c].numpy()
        fm = (fm - fm.min()) / (fm.max() - fm.min() + 1e-8)
        axes_flat[n_cols + c].imshow(fm, cmap='viridis')
        axes_flat[n_cols + c].set_title(f'Ch {c}', fontsize=6)
        axes_flat[n_cols + c].axis('off')
    for c in range(n_cols + n_ch, len(axes_flat)): axes_flat[c].axis('off')

    plt.suptitle(f'Section 2.4: Feature Maps — {layer_name} ({feat.shape[0]} channels, showing {n_ch})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    wandb.log({f'sec2.4/feature_maps_{layer_name}': fig_to_wandb(fig)})

print('Section 2.4 done.')

---
## Section 2.5 — Detection Table: Bbox Predictions + IoU

In [ ]:
# 2.5: Log detection table with 10 images, GT vs Pred bboxes, IoU
from sklearn.metrics import f1_score

detection_images = []
iou_list = []
count = 0

for batch in val_loader:
    imgs = batch['image'].to(device)
    bboxes_gt = batch['bbox']
    labels = batch['label']

    with torch.no_grad():
        bboxes_pred = loc_model(imgs).cpu()
        cls_logits = cls_model(imgs)
        cls_probs = torch.softmax(cls_logits, dim=1)
        confs, preds = cls_probs.max(dim=1)

    ious = compute_iou(bboxes_pred, bboxes_gt)

    for i in range(imgs.size(0)):
        if count >= 12: break
        img_np = denormalize_image(imgs[i])
        gt = bboxes_gt[i].numpy()
        pr = bboxes_pred[i].numpy()
        iou_val = ious[i].item()
        conf_val = confs[i].item()

        fig, ax = plt.subplots(1, 1, figsize=(5, 5))
        ax.imshow(img_np)
        # GT box (green)
        gt_rect = patches.Rectangle((gt[0]-gt[2]/2, gt[1]-gt[3]/2), gt[2], gt[3],
                                     linewidth=2, edgecolor='green', facecolor='none', linestyle='-')
        ax.add_patch(gt_rect)
        # Pred box (red)
        pr_rect = patches.Rectangle((pr[0]-pr[2]/2, pr[1]-pr[3]/2), pr[2], pr[3],
                                     linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
        ax.add_patch(pr_rect)
        ax.set_title(f'IoU={iou_val:.3f} | Conf={conf_val:.2f}\nGT(green) Pred(red)', fontsize=10)
        ax.axis('off')
        plt.tight_layout()

        detection_images.append(fig_to_wandb(fig,
            caption=f'IoU={iou_val:.3f}, Conf={conf_val:.2f}, Pred={class_names[preds[i].item()]}'))
        iou_list.append(iou_val)
        count += 1
    if count >= 12: break

wandb.log({'sec2.5/detection_table': detection_images})
print(f'Section 2.5 done. Mean IoU: {np.mean(iou_list):.4f}, Acc@0.5: {np.mean([i>=0.5 for i in iou_list])*100:.0f}%')

---
## Section 2.6 — Segmentation: Dice vs Pixel Accuracy

In [ ]:
# 2.6: 5 sample images with Original / GT trimap / Predicted trimap
colormap = np.array([[0, 200, 0], [40, 40, 40], [255, 255, 0]], dtype=np.uint8)  # FG/BG/boundary
seg_images = []
dice_scores, pixel_accs = [], []
count = 0

for batch in val_loader:
    imgs = batch['image'].to(device)
    masks_gt = batch['mask']

    with torch.no_grad():
        masks_pred = seg_model(imgs).argmax(1).cpu()

    for i in range(imgs.size(0)):
        if count >= 5: break
        img_np = denormalize_image(imgs[i])
        gt_mask = masks_gt[i].numpy()
        pr_mask = masks_pred[i].numpy()

        dice = compute_dice_score(masks_pred[i:i+1], masks_gt[i:i+1])
        pa = compute_pixel_accuracy(masks_pred[i:i+1], masks_gt[i:i+1])
        dice_scores.append(dice)
        pixel_accs.append(pa)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(img_np); axes[0].set_title('Original'); axes[0].axis('off')
        axes[1].imshow(colormap[gt_mask]); axes[1].set_title('GT Trimap'); axes[1].axis('off')
        axes[2].imshow(colormap[pr_mask]); axes[2].set_title(f'Predicted\nDice={dice:.3f}, PixAcc={pa:.3f}'); axes[2].axis('off')
        plt.suptitle(f'Sample {count+1}', fontsize=13, fontweight='bold')
        plt.tight_layout()

        seg_images.append(fig_to_wandb(fig, caption=f'Dice={dice:.3f}, PixAcc={pa:.3f}'))
        count += 1
    if count >= 5: break

wandb.log({'sec2.6/segmentation_samples': seg_images})

# Full val set: Dice vs Pixel Accuracy per epoch (from W&B runs)
seg_run = find_runs(tag='freeze-partial')
if seg_run:
    h = seg_run[0].history(pandas=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    if 'val/dice' in h.columns:
        ax.plot(h['val/dice'].dropna().values, label='Dice Score', color='#2196F3', linewidth=2)
    if 'val/pixel_accuracy' in h.columns:
        ax.plot(h['val/pixel_accuracy'].dropna().values, label='Pixel Accuracy', color='#FF9800', linewidth=2)
    ax.set_title('Section 2.6: Dice Score vs Pixel Accuracy During Training', fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Score')
    ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    wandb.log({'sec2.6/dice_vs_pixel_accuracy': fig_to_wandb(fig)})

print(f'Section 2.6 done. Avg Dice={np.mean(dice_scores):.4f}, Avg PixAcc={np.mean(pixel_accs):.4f}')

---
## Section 2.8 — Meta-Analysis: Comprehensive Plots

In [ ]:
# 2.8: Combined metric plots across all tasks
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Classification: best run
cls_runs = find_runs(name_contains='pretrained=True')
if not cls_runs: cls_runs = find_runs(tag='kaggle-pretrained')
if not cls_runs: cls_runs = find_runs(tag='classification')[:1]
if cls_runs:
    h = cls_runs[0].history(pandas=True)
    if 'train/loss' in h.columns:
        axes[0].plot(h['train/loss'].dropna().values, label='Train Loss', linestyle='--', alpha=0.7)
    if 'val/loss' in h.columns:
        axes[0].plot(h['val/loss'].dropna().values, label='Val Loss')
    axes[0].set_title('Classification (Best Run)'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Segmentation: partial freeze
seg_runs = find_runs(tag='freeze-partial')
if seg_runs:
    h = seg_runs[0].history(pandas=True)
    if 'train/loss' in h.columns:
        axes[1].plot(h['train/loss'].dropna().values, label='Train Loss', linestyle='--', alpha=0.7)
    if 'val/loss' in h.columns:
        axes[1].plot(h['val/loss'].dropna().values, label='Val Loss')
    axes[1].set_title('Segmentation (Partial Freeze)'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# Shared encoder UNet
unet_runs = find_runs(tag='shared-encoder')
if unet_runs:
    h = unet_runs[0].history(pandas=True)
    for col in ['val/dice', 'val/mean_dice']:
        if col in h.columns:
            axes[2].plot(h[col].dropna().values, label='Val Dice', color='#4CAF50')
            break
    axes[2].set_title('UNet Shared Encoder'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Dice')
    axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

plt.suptitle('Section 2.8: Training History Across All Tasks', fontsize=14, fontweight='bold')
plt.tight_layout()
wandb.log({'sec2.8/meta_analysis_all_tasks': fig_to_wandb(fig)})

# Final scores summary bar chart
fig, ax = plt.subplots(figsize=(8, 5))
tasks = ['Classification\n(Macro-F1)', 'Localization\n(Acc@IoU=0.5)', 'Segmentation\n(Dice)']
scores = [0.9325, 0.918, 0.8577]  # from Gradescope/Kaggle baseline
thresholds = [0.80, 0.60, 0.30]
colors_bar = ['#2196F3', '#4CAF50', '#FF9800']
bars = ax.bar(range(3), scores, color=colors_bar, alpha=0.85, width=0.5)
for i, (thr, sc) in enumerate(zip(thresholds, scores)):
    ax.plot([i-0.3, i+0.3], [thr, thr], 'r--', linewidth=2)
    ax.text(i, sc+0.02, f'{sc:.3f}', ha='center', fontsize=13, fontweight='bold')
ax.set_xticks(range(3)); ax.set_xticklabels(tasks, fontsize=11)
ax.set_ylabel('Score'); ax.set_ylim(0, 1.1)
ax.set_title('Final Pipeline Scores vs Thresholds', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
wandb.log({'sec2.8/final_scores': fig_to_wandb(fig)})
print('Section 2.8 done.')

In [ ]:
wandb.finish()
print('='*55)
print('  ALL REPORT PANELS LOGGED TO W&B!')
print('='*55)
print()
print('Panels logged:')
for sec in ['sec2.1/activation_distributions', 'sec2.1/bn_comparison_curves',
            'sec2.2/dropout_comparison', 'sec2.3/transfer_learning',
            'sec2.4/feature_maps_first_conv', 'sec2.4/feature_maps_last_conv',
            'sec2.5/detection_table', 'sec2.6/segmentation_samples',
            'sec2.6/dice_vs_pixel_accuracy', 'sec2.8/meta_analysis_all_tasks',
            'sec2.8/final_scores']:
    print(f'  {sec}')